GRU

In [1]:
import duckdb
import numpy as np
import pandas as pd
import pandas_ta as ta
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
import tensorflow as tf
print(tf.sysconfig.get_build_info())
print("GPU:", tf.config.list_physical_devices('GPU'))

OrderedDict({'is_cuda_build': False, 'is_rocm_build': False, 'is_tensorrt_build': False, 'msvcp_dll_names': 'msvcp140.dll,msvcp140_1.dll'})
GPU: []


In [3]:
# GPU 인식 여부 확인
print(tf.__version__)
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

2.16.1
Num GPUs Available:  0


In [2]:
con = duckdb.connect(database='D:/Assets/BinanceFuturesData/binancefuturesdata.duckdb', read_only=True)

In [3]:
data_query = "SELECT symbol, epoch_ms(timestamp) as time, open, high, low, close, volume FROM quote " \
"WHERE symbol = 'BTCUSDT' " \
"and epoch_ms(timestamp) >= '2025-01-01 00:00:00' " \
"and epoch_ms(timestamp) < '2025-12-01 00:00:00' " \
"ORDER BY timestamp;"
df = con.execute(data_query).fetchdf()

con.close()

In [4]:
df

,symbol,time,open,high,low,close,volume
0,BTCUSDT,2025-01-01 00:00:00,93548.8,93599.9,93514.2,93599.9,71.187
1,BTCUSDT,2025-01-01 00:01:00,93599.9,93637.7,93577.6,93637.7,39.526
2,BTCUSDT,2025-01-01 00:02:00,93637.7,93690.0,93614.2,93688.5,94.805
3,BTCUSDT,2025-01-01 00:03:00,93688.5,93688.5,93626.4,93664.6,45.566
4,BTCUSDT,2025-01-01 00:04:00,93664.5,93668.3,93626.3,93648.4,90.520
...,...,...,...,...,...,...,...
480955,BTCUSDT,2025-11-30 23:55:00,90353.4,90402.0,90347.8,90402.0,102.526
480956,BTCUSDT,2025-11-30 23:56:00,90401.9,90432.3,90364.3,90364.3,117.141
480957,BTCUSDT,2025-11-30 23:57:00,90364.3,90406.0,90333.3,90372.2,128.229
480958,BTCUSDT,2025-11-30 23:58:00,90372.2,90372.2,90300.0,90316.9,184.376


In [4]:
# Datetime 컬럼을 인덱스로 설정
df['time'] = pd.to_datetime(df['time'])
df = df.set_index('time')

In [5]:
# 결측치 처리
df.dropna(inplace=True)

특징 공학 및 스케일링 (Feature Engineering & Scaling)

In [6]:
# 기술적 지표 추가
df.ta.cci(append=True)
df.ta.rsi(append=True)
df.ta.macd(append=True)
df.ta.bbands(append=True)
df.dropna(inplace=True)

# --- 데이터 누수 방지를 위한 수정된 전처리 과정 ---

# 1. 예측 대상(Target) 정의: 다음 1분 뒤의 종가 수익률
df['Target_Return'] = df['close'].pct_change().shift(-1) * 100
df.dropna(inplace=True)

# 2. 특성(X)과 타겟(y) 분리
# Target_Return은 예측 대상이므로 특성(X)에서 제외해야 합니다.
feature_cols = [col for col in df.columns if col != 'Target_Return' and df[col].dtype in [np.int64, np.float64]]
target_col = 'Target_Return'

X_df = df[feature_cols]
y_df = df[[target_col]]

# 3. 훈련(Train) / 테스트(Test) 데이터 분리 (스케일링 전)
train_size = int(len(df) * 0.8)
X_train_df, X_test_df = X_df.iloc[:train_size], X_df.iloc[train_size:]
y_train_df, y_test_df = y_df.iloc[:train_size], y_df.iloc[train_size:]

# 4. 스케일링 (훈련 데이터에만 fit)
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# 훈련 데이터로 스케일러 학습
X_train_scaled = scaler_X.fit_transform(X_train_df)
y_train_scaled = scaler_y.fit_transform(y_train_df)

# 학습된 스케일러로 테스트 데이터 변환
X_test_scaled = scaler_X.transform(X_test_df)
y_test_scaled = scaler_y.transform(y_test_df)

시퀀스 데이터 변환

In [7]:
def create_sequences(X_data, y_data, look_back):
    X, y = [], []
    for i in range(len(X_data) - look_back):
        X.append(X_data[i:(i + look_back), :])
        y.append(y_data[i + look_back])
    return np.array(X), np.array(y)

In [8]:
look_back = 60 # 과거 60분 데이터 사용

# 훈련 데이터 시퀀스 생성
X_train, y_train = create_sequences(X_train_scaled, y_train_scaled, look_back)

# 테스트 데이터 시퀀스 생성
X_test, y_test = create_sequences(X_test_scaled, y_test_scaled, look_back)

In [9]:
# 생성된 데이터 차원 확인
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (384680, 60, 15)
y_train shape: (384680, 1)
X_test shape: (96126, 60, 15)
y_test shape: (96126, 1)


GRU 모델 구축 및 컴파일

In [10]:
model = Sequential()

# GRU 레이어의 유닛 수를 늘려 모델의 용량 증대 (4 -> 50)
model.add(GRU(units=50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.2))

# 두 번째 GRU 레이어
model.add(GRU(units=50))
model.add(Dropout(0.2))

# 출력 레이어
model.add(Dense(units=1, activation='linear'))

C:\Users\Gaten\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [11]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 60, 50)         │        10,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 50)             │        15,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,401 (99.22 KB)

 Trainable params: 25,401 (99.22 KB)

 Non-trainable params: 0 (0.00 B)

모델 학습 및 평가

In [12]:
# 검증 손실이 10 에포크 동안 개선되지 않으면 학습 중단
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Epochs를 50으로 늘려 충분히 학습
history = model.fit(
    X_train, y_train, 
    epochs=50, 
    batch_size=256, 
    validation_data=(X_test, y_test), 
    callbacks=[early_stop], 
    verbose=1
)

Epoch 1/50
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 92s 60ms/step - loss: 0.0020 - mae: 0.0311 - val_loss: 3.4605e-04 - val_mae: 0.0122
Epoch 2/50
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 113s 75ms/step - loss: 5.0819e-04 - mae: 0.0165 - val_loss: 3.3319e-04 - val_mae: 0.0117
Epoch 3/50
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 115s 77ms/step - loss: 3.0645e-04 - mae: 0.0116 - val_loss: 3.3160e-04 - val_mae: 0.0117
Epoch 4/50
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 117s 78ms/step - loss: 2.7953e-04 - mae: 0.0107 - val_loss: 3.2158e-04 - val_mae: 0.0113
Epoch 5/50
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 119s 79ms/step - loss: 2.7633e-04 - mae: 0.0105 - val_loss: 3.2699e-04 - val_mae: 0.0115
Epoch 6/50
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 119s 79ms/step - loss: 2.7343e-04 - mae: 0.0104 - val_loss: 3.2540e-04 - val_mae: 0.0114
Epoch 7/50
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 121s 80ms/step - loss: 2.7089e-04 - mae: 0.0103 - val_loss: 3.2143e-04 - val_mae: 0.0112
Epoch 8/50
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 105s 70ms/step - loss: 2.6973e-04 - mae: 0.010

In [1]:
# 모델 최종 평가
loss, mae = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Loss (MSE): {loss:.8f}') # 데이터가 작아졌으므로 소수점 늘림
print(f'Test MAE: {mae:.8f}')

NameError: name 'model' is not defined

결과 분석 및 시그널 생성

In [15]:
# 테스트 데이터에 대한 예측
y_pred_scaled = model.predict(X_test)

3005/3005 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step


In [16]:
# --- 예측 결과 역변환 ---
# y 스케일러(scaler_y)를 사용하여 스케일링된 예측값과 실제값을 원래의 % 수익률로 되돌립니다.
y_pred_actual = scaler_y.inverse_transform(y_pred_scaled)
y_test_actual = scaler_y.inverse_transform(y_test)

# --- 결과 비교 데이터프레임 생성 ---
# 테스트 기간의 실제 종가와 예측 결과를 비교하기 위해 데이터프레임을 만듭니다.
# y_test_df 에서 y_test에 해당하는 부분만 추출합니다. (시퀀스 생성 시 앞부분 look_back 만큼 데이터가 손실됨)
results_df = y_test_df.iloc[look_back:].copy()
results_df['Predicted_Return'] = y_pred_actual
results_df.rename(columns={'Target_Return': 'Actual_Return'}, inplace=True)

print("--- 실제 수익률 vs 예측 수익률 ---")
print(results_df.head())

--- 실제 수익률 vs 예측 수익률 ---
                     Actual_Return  Predicted_Return
time                                                
2025-09-25 05:49:00       0.022387          0.001970
2025-09-25 05:50:00       0.033304          0.002080
2025-09-25 05:51:00      -0.001342          0.002140
2025-09-25 05:52:00      -0.031145          0.002290
2025-09-25 05:53:00      -0.010474          0.002277


In [17]:
# --- 트레이딩 시그널 생성 ---
# 진입 임계값 (threshold) 설정 (예: 0.1% 이상 변동 예측 시 진입)
threshold = 0.05

# 시그널 생성: 1 = 롱(Long), -1 = 숏(Short), 0 = 관망(Hold)
results_df['signal'] = 0
results_df.loc[results_df['Predicted_Return'] > threshold, 'signal'] = 1
results_df.loc[results_df['Predicted_Return'] < -threshold, 'signal'] = -1

print(f"\n--- 시그널 생성 결과 (Threshold = {threshold}%) ---")
print(f"롱 시그널 수: {np.sum(results_df['signal'] == 1)}")
print(f"숏 시그널 수: {np.sum(results_df['signal'] == -1)}")
print(f"관망 수: {np.sum(results_df['signal'] == 0)}")


# --- 백테스팅 시뮬레이션 ---
initial_capital = 100000  # 초기 자본
transaction_cost_rate = 0.0005  # 0.05% (매수/매도 각각)

capital = initial_capital
portfolio_value_history = [initial_capital]

for index, row in results_df.iterrows():
    signal = row['signal']
    actual_return = row['Actual_Return'] / 100.0  # %를 소수점으로 변환

    if signal == 1:  # 롱 포지션
        # 매수 (거래 비용 적용)
        # 다음 캔들의 실제 수익률을 적용
        capital *= (1 + actual_return)  # 실제 수익률 반영
        capital *= (1 - transaction_cost_rate)  # 매수 수수료
        capital *= (1 - transaction_cost_rate)  # 매도 수수료 (같은 캔들 내 청산 가정)
    elif signal == -1:  # 숏 포지션
        # 숏 매도 (거래 비용 적용)
        # 다음 캔들의 실제 수익률을 역으로 적용
        capital *= (1 - actual_return)  # 실제 수익률 반영 (숏이므로 반대)
        capital *= (1 - transaction_cost_rate)  # 매도 수수료
        capital *= (1 - transaction_cost_rate)  # 매수 수수료 (같은 캔들 내 청산 가정)

    portfolio_value_history.append(capital)

portfolio_series = pd.Series(portfolio_value_history, index=[results_df.index[0]] + list(results_df.index))
cumulative_returns = (portfolio_series.iloc[-1] / initial_capital - 1) * 100

print(f"\n--- 백테스팅 결과 ---")
print(f"초기 자본: {initial_capital:,.0f}")
print(f"최종 자본: {portfolio_series.iloc[-1]:,.2f}")
print(f"총 누적 수익률: {cumulative_returns:.2f}%")

# 결과 시각화 (선택 사항)
# import matplotlib.pyplot as plt
# plt.figure(figsize=(12, 6))
# portfolio_series.plot(title='Cumulative Portfolio Value Over Time')
# plt.xlabel('Time')
# plt.ylabel('Portfolio Value')
# plt.grid(True)
# plt.show()


--- 시그널 생성 결과 (Threshold = 0.05%) ---
롱 시그널 수: 8
숏 시그널 수: 0
관망 수: 96122

--- 예측 정확도 ---
롱 포지션 예측 정확도: 62.50%
숏 포지션 예측 정확도: nan%
